# PHÂN TÍCH ĐÁNH GIÁ KHÁCH HÀNG — 3 MÔ HÌNH ML CƠ BẢN
**Bài toán:** Phân loại cảm xúc/khuyến nghị (Binary Classification)
---
**3 Mô hình ML:** Multinomial Naive Bayes, Logistic Regression, Random Forest.

In [1]:
import numpy as np
import pandas as pd
import joblib, os
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score
from sklearn.naive_bayes import MultinomialNB
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier

MODEL_DIR = os.path.join('..', 'models')
data = np.load(os.path.join(MODEL_DIR, 'tfidf_data.npz'))
X_train, y_train = data['X_train'], data['y_train']
X_val, y_val = data['X_val'], data['y_val']
X_test, y_test = data['X_test'], data['y_test']
print('Loaded Data:', X_train.shape, X_val.shape, X_test.shape)

Loaded Data: (15835, 3000) (3393, 3000) (3394, 3000)


In [2]:
def evaluate(model, X_train, y_train, X_test, y_test, name):
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    y_prob = model.predict_proba(X_test)[:, 1] if hasattr(model, 'predict_proba') else None
    
    acc = accuracy_score(y_test, y_pred)
    prec = precision_score(y_test, y_pred, zero_division=0)
    rec = recall_score(y_test, y_pred, zero_division=0)
    f1 = f1_score(y_test, y_pred, zero_division=0)
    auc = roc_auc_score(y_test, y_prob) if y_prob is not None else 0.5
    
    return {
        'name': name,
        'model': model,
        'Accuracy': acc,
        'Precision': prec,
        'Recall': rec,
        'F1-Score': f1,
        'AUC-ROC': auc,
        'y_prob': y_prob
    }


In [3]:
res_nb = evaluate(MultinomialNB(), X_train, y_train, X_test, y_test, 'Multinomial Naive Bayes')
res_lr = evaluate(LogisticRegression(max_iter=500, random_state=42), X_train, y_train, X_test, y_test, 'Logistic Regression')
res_rf = evaluate(RandomForestClassifier(n_estimators=100, max_depth=15, n_jobs=-1, random_state=42), X_train, y_train, X_test, y_test, 'Random Forest')

res_list = [res_nb, res_lr, res_rf]
df_comp = pd.DataFrame([{k:v for k,v in r.items() if k not in ['model', 'y_prob']} for r in res_list])
print(df_comp)

for r in res_list:
    joblib.dump(r['model'], os.path.join(MODEL_DIR, f"cb_{r['name'].lower().replace(' ', '_')}.pkl"))
print('Saved models.')

                      name  Accuracy  Precision    Recall  F1-Score   AUC-ROC
0  Multinomial Naive Bayes  0.853565   0.853251  0.991724  0.917291  0.922217
1      Logistic Regression  0.886270   0.898966  0.970133  0.933195  0.933118
2            Random Forest  0.819976   0.819764  1.000000  0.900956  0.897031
Saved models.


### Nhận xét kết quả ML
- Bảng kết quả cần được đọc đồng thời qua Accuracy, F1-Score và AUC-ROC; với bài toán nhị phân, F1 phản ánh cân bằng Precision–Recall tốt hơn Accuracy khi hai lớp không hoàn toàn cân bằng.
- Logistic Regression thường là mốc đối chứng phù hợp cho TF-IDF vì đặc trưng văn bản có chiều cao và thưa; Naive Bayes ưu tiên tốc độ, còn Random Forest có thể khó khai thác hiệu quả toàn bộ không gian từ vựng.